<a href="https://colab.research.google.com/github/Sattvikannadatha/LLM-Ticket/blob/main/sattvik_hard_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 wh

In [10]:
from ortools.sat.python import cp_model
import time
import math

def solve_tz_magic_square(max_cell_val=1500):
    print("=========================================================")
    print(" INITIATING 3x3 T-VECTOR & Z-VECTOR MAGIC ENGINE")
    print("=========================================================\n")

    model = cp_model.CpModel()
    n = 3

    # 1. Grid Variables
    grid = []
    for i in range(n):
        row = [model.NewIntVar(1, max_cell_val, f'x_{i}_{j}') for j in range(n)]
        grid.append(row)

    flat_grid = [cell for row in grid for cell in row]
    model.AddAllDifferent(flat_grid)

    # 2. Magic Constant (M)
    M = model.NewIntVar(15, n * max_cell_val, 'M')

    # Standard Magic Square Constraints
    for i in range(n):
        model.Add(sum(grid[i]) == M) # Rows
        model.Add(sum(grid[j][i] for j in range(n)) == M) # Cols

    model.Add(sum(grid[i][i] for i in range(n)) == M) # Main Diag
    model.Add(sum(grid[i][n-1-i] for i in range(n)) == M) # Anti Diag

    # 3. Define the Geometric Vectors
    # T-Vectors (Outer edge + perpendicular center stem = 5 cells)
    t_up = grid[0] + [grid[1][1], grid[2][1]]
    t_down = grid[2] + [grid[1][1], grid[0][1]]
    t_left = [grid[0][0], grid[1][0], grid[2][0], grid[1][1], grid[1][2]]
    t_right = [grid[0][2], grid[1][2], grid[2][2], grid[1][1], grid[1][0]]

    # Z-Vector (Top row + anti-diagonal + bottom row = 7 distinct cells)
    # Avoiding double-counting the corners at (0,2) and (2,0)
    z_vec = [grid[0][0], grid[0][1], grid[0][2],
             grid[1][1],
             grid[2][0], grid[2][1], grid[2][2]]

    # 4. Perfect Square Constraints
    max_possible_sum = 7 * max_cell_val
    squares = [i**2 for i in range(2, int(math.isqrt(max_possible_sum)) + 1)]
    squares_domain = cp_model.Domain.FromValues(squares)

    vector_sums = []
    vec_names = ['T_Up', 'T_Down', 'T_Left', 'T_Right', 'Z_Vector']
    vectors = [t_up, t_down, t_left, t_right, z_vec]

    for name, vec in zip(vec_names, vectors):
        v_sum = model.NewIntVarFromDomain(squares_domain, f'sum_{name}')
        model.Add(sum(vec) == v_sum)
        vector_sums.append(v_sum)

    # CLOSING THE LOOPHOLE: All 5 vector sums must be strictly unique
    model.AddAllDifferent(vector_sums)

    # 5. Solver Setup
    solver = cp_model.CpSolver()
    solver.parameters.num_search_workers = 8
    solver.parameters.max_time_in_seconds = 4400.0

    print(f"[*] Search Space: Max cell value = {max_cell_val}")
    print(f"[*] Valid Perfect Square Targets: {len(squares)}")
    print("\n[>>] EXECUTING CP-SAT ENGINE...")

    start_time = time.time()
    status = solver.Solve(model)

    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f"\n✅ MATRIX SOLVED IN {time.time() - start_time:.2f} SECONDS!\n")
        print(f"MAGIC CONSTANT (M) = {solver.Value(M)}")
        print("=========================================================")
        for i in range(n):
            row_vals = [solver.Value(grid[i][j]) for j in range(n)]
            print(f"  {row_vals}")
        print("=========================================================")
        print("VECTOR SUMS (Unique Perfect Squares):")
        for name, v_var in zip(vec_names, vector_sums):
            val = solver.Value(v_var)
            base = int(math.isqrt(val))
            print(f"  {name:<8} : {val:<5} -> {base}²")
    else:
        print(f"\n❌ Search terminated in {time.time() - start_time:.2f}s (Infeasible or Timeout).")
        print("Try increasing 'max_cell_val' to give the engine more numerical building blocks.")

if __name__ == '__main__':
    solve_tz_magic_square(max_cell_val=50500)

 INITIATING 3x3 T-VECTOR & Z-VECTOR MAGIC ENGINE

[*] Search Space: Max cell value = 50500
[*] Valid Perfect Square Targets: 593

[>>] EXECUTING CP-SAT ENGINE...

❌ Search terminated in 0.84s (Infeasible or Timeout).
Try increasing 'max_cell_val' to give the engine more numerical building blocks.


In [11]:
from ortools.sat.python import cp_model
import math
import time

def solve_envelope_magic_square(n=5, max_cell_val=800):
    print("=========================================================")
    print(f" INITIATING {n}x{n} ENVELOPE GEOMETRY ENGINE")
    print("=========================================================\n")

    model = cp_model.CpModel()

    # 1. Grid Variables
    grid = []
    for i in range(n):
        row = [model.NewIntVar(1, max_cell_val, f'x_{i}_{j}') for j in range(n)]
        grid.append(row)

    flat_grid = [cell for row in grid for cell in row]
    model.AddAllDifferent(flat_grid)

    # 2. Base Magic Constraints
    max_magic_sum = n * max_cell_val
    M = model.NewIntVar(15, max_magic_sum, 'M')

    for i in range(n):
        model.Add(sum(grid[i]) == M) # Rows
        model.Add(sum(grid[j][i] for j in range(n)) == M) # Cols

    model.Add(sum(grid[i][i] for i in range(n)) == M) # Main Diag
    model.Add(sum(grid[i][n-1-i] for i in range(n)) == M) # Anti Diag

    # 3. Extract the 4 Envelope Triangles
    # Using geometric coordinates:
    # Top: i <= j AND i + j <= n - 1
    # Bot: i >= j AND i + j >= n - 1
    # Lft: j <= i AND i + j <= n - 1
    # Rgt: j >= i AND i + j >= n - 1

    t_top = []
    t_bot = []
    t_left = []
    t_right = []

    for i in range(n):
        for j in range(n):
            if i <= j and i + j <= n - 1:
                t_top.append(grid[i][j])
            if i >= j and i + j >= n - 1:
                t_bot.append(grid[i][j])
            if j <= i and i + j <= n - 1:
                t_left.append(grid[i][j])
            if j >= i and i + j >= n - 1:
                t_right.append(grid[i][j])

    # 4. Perfect Square Constraints
    # The max sum for 9 cells is 9 * max_cell_val
    max_triangle_sum = 9 * max_cell_val
    squares = [i**2 for i in range(2, int(math.isqrt(max_triangle_sum)) + 1)]
    domain_squares = cp_model.Domain.FromValues(squares)

    print(f"[*] Base Geometry: {len(t_top)} cells per envelope flap.")
    print(f"[*] Maximum possible flap sum: {max_triangle_sum}")
    print(f"[*] Valid Perfect Square Targets: {len(squares)}")

    t_top_sum = model.NewIntVarFromDomain(domain_squares, 't_top_sum')
    t_bot_sum = model.NewIntVarFromDomain(domain_squares, 't_bot_sum')
    t_left_sum = model.NewIntVarFromDomain(domain_squares, 't_left_sum')
    t_right_sum = model.NewIntVarFromDomain(domain_squares, 't_right_sum')

    model.Add(sum(t_top) == t_top_sum)
    model.Add(sum(t_bot) == t_bot_sum)
    model.Add(sum(t_left) == t_left_sum)
    model.Add(sum(t_right) == t_right_sum)

    # Force the 4 triangles to be STRICTLY UNIQUE perfect squares
    # This prevents the solver from using symmetrical mirror tricks.
    model.AddAllDifferent([t_top_sum, t_bot_sum, t_left_sum, t_right_sum])

    # 5. Solver Setup
    solver = cp_model.CpSolver()
    solver.parameters.num_search_workers = 8
    solver.parameters.max_time_in_seconds = 300.0
    solver.parameters.log_search_progress = True

    print("\n[>>] FOLDING THE MATRIX...")
    start_time = time.time()
    status = solver.Solve(model)

    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f"\n✅ ENVELOPE SEALED IN {time.time() - start_time:.2f} SECONDS!\n")
        print(f"MAGIC CONSTANT (M) = {solver.Value(M)}")
        print("=" * 65)
        for i in range(n):
            row_vals = [solver.Value(grid[i][j]) for j in range(n)]
            print(f"  {row_vals}")

        print("\nENVELOPE TRIANGLE SUMS (Unique Perfect Squares):")
        print("-" * 65)
        sums = {
            "Top Flap"   : solver.Value(t_top_sum),
            "Bottom Flap": solver.Value(t_bot_sum),
            "Left Flap"  : solver.Value(t_left_sum),
            "Right Flap" : solver.Value(t_right_sum)
        }
        for name, val in sums.items():
            base = int(math.isqrt(val))
            print(f"  {name:<12} : {val:<5} -> {base}²")
        print("=" * 65)
    else:
        print(f"\n❌ Search completed in {time.time() - start_time:.2f}s with status INFEASIBLE.")
        print("Try increasing 'max_cell_val' to give the engine a larger numerical pool.")

if __name__ == '__main__':
    # Using 800 gives the solver a massive playground to find 4 unique squares.
    solve_envelope_magic_square(n=5, max_cell_val=800)

 INITIATING 5x5 ENVELOPE GEOMETRY ENGINE

[*] Base Geometry: 9 cells per envelope flap.
[*] Maximum possible flap sum: 7200
[*] Valid Perfect Square Targets: 83

[>>] FOLDING THE MATRIX...

✅ ENVELOPE SEALED IN 136.90 SECONDS!

MAGIC CONSTANT (M) = 807
  [8, 197, 5, 401, 196]
  [740, 6, 7, 3, 51]
  [1, 2, 402, 13, 389]
  [4, 152, 279, 296, 76]
  [54, 450, 114, 94, 95]

ENVELOPE TRIANGLE SUMS (Unique Perfect Squares):
-----------------------------------------------------------------
  Top Flap     : 1225  -> 35²
  Bottom Flap  : 1936  -> 44²
  Left Flap    : 1369  -> 37²
  Right Flap   : 1521  -> 39²


In [12]:
import math

def visualize_envelope_matrix():
    # The winning matrix hardcoded
    grid = [
        [  8, 197,   5, 401, 196],
        [740,   6,   7,   3,  51],
        [  1,   2, 402,  13, 389],
        [  4, 152, 279, 296,  76],
        [ 54, 450, 114,  94,  95]
    ]
    n = 5

    print("=========================================================")
    print(" 5x5 ENVELOPE GEOMETRY: X-RAY VISUALIZER")
    print("=========================================================\n")

    # Define the geometric coordinate rules for the 4 flaps
    flaps = {
        "TOP FLAP":    lambda i, j: i <= j and i + j <= n - 1,
        "BOTTOM FLAP": lambda i, j: i >= j and i + j >= n - 1,
        "LEFT FLAP":   lambda i, j: j <= i and i + j <= n - 1,
        "RIGHT FLAP":  lambda i, j: j >= i and i + j >= n - 1
    }

    for flap_name, condition in flaps.items():
        print(f"--- {flap_name} ---")

        flap_numbers = []

        # 1. Draw the visual matrix mask
        for i in range(n):
            row_str = ""
            for j in range(n):
                if condition(i, j):
                    val = grid[i][j]
                    flap_numbers.append(val)
                    row_str += f"{val:>5}" # Align the numbers nicely
                else:
                    row_str += "    ."  # Mask out the non-flap numbers
            print(row_str)

        # 2. Show the mathematical working
        total_sum = sum(flap_numbers)
        base = int(math.isqrt(total_sum))

        # Format the addition string
        addition_str = " + ".join(str(num) for num in flap_numbers)

        print("\nMathematical Working:")
        print(f"Cells: {addition_str}")
        print(f"Total: {total_sum} -> {base}² (Perfect Square)\n")
        print("=" * 57 + "\n")

if __name__ == '__main__':
    visualize_envelope_matrix()

 5x5 ENVELOPE GEOMETRY: X-RAY VISUALIZER

--- TOP FLAP ---
    8  197    5  401  196
    .    6    7    3    .
    .    .  402    .    .
    .    .    .    .    .
    .    .    .    .    .

Mathematical Working:
Cells: 8 + 197 + 5 + 401 + 196 + 6 + 7 + 3 + 402
Total: 1225 -> 35² (Perfect Square)


--- BOTTOM FLAP ---
    .    .    .    .    .
    .    .    .    .    .
    .    .  402    .    .
    .  152  279  296    .
   54  450  114   94   95

Mathematical Working:
Cells: 402 + 152 + 279 + 296 + 54 + 450 + 114 + 94 + 95
Total: 1936 -> 44² (Perfect Square)


--- LEFT FLAP ---
    8    .    .    .    .
  740    6    .    .    .
    1    2  402    .    .
    4  152    .    .    .
   54    .    .    .    .

Mathematical Working:
Cells: 8 + 740 + 6 + 1 + 2 + 402 + 4 + 152 + 54
Total: 1369 -> 37² (Perfect Square)


--- RIGHT FLAP ---
    .    .    .    .  196
    .    .    .    3   51
    .    .  402   13  389
    .    .    .  296   76
    .    .    .    .   95

Mathematical Working:
Cells

In [13]:
def verify_base_magic_square():
    # The winning matrix hardcoded
    grid = [
        [  8, 197,   5, 401, 196],
        [740,   6,   7,   3,  51],
        [  1,   2, 402,  13, 389],
        [  4, 152, 279, 296,  76],
        [ 54, 450, 114,  94,  95]
    ]
    n = 5
    expected_M = 807

    print("=========================================================")
    print(" 5x5 MAGIC SQUARE: BASE VECTOR VERIFICATION")
    print("=========================================================\n")

    # 1. Verify Rows
    print("1. ROW VECTORS")
    for i in range(n):
        row_sum = sum(grid[i])
        equation = " + ".join(f"{val:>3}" for val in grid[i])
        print(f"   Row {i}: {equation} = {row_sum}")

    # 2. Verify Columns
    print("\n2. COLUMN VECTORS")
    for j in range(n):
        col = [grid[i][j] for i in range(n)]
        col_sum = sum(col)
        equation = " + ".join(f"{val:>3}" for val in col)
        print(f"   Col {j}: {equation} = {col_sum}")

    # 3. Verify Diagonals
    print("\n3. DIAGONAL VECTORS")
    main_diag = [grid[i][i] for i in range(n)]
    main_diag_sum = sum(main_diag)
    eq_main = " + ".join(f"{val:>3}" for val in main_diag)
    print(f"   Main Diag (\\): {eq_main} = {main_diag_sum}")

    anti_diag = [grid[i][n-1-i] for i in range(n)]
    anti_diag_sum = sum(anti_diag)
    eq_anti = " + ".join(f"{val:>3}" for val in anti_diag)
    print(f"   Anti Diag (/): {eq_anti} = {anti_diag_sum}")

    # Final Check
    print("\n=========================================================")
    print(f" STATUS: ALL 12 VECTORS CONFIRMED = {expected_M}")
    print("=========================================================")

if __name__ == '__main__':
    verify_base_magic_square()

 5x5 MAGIC SQUARE: BASE VECTOR VERIFICATION

1. ROW VECTORS
   Row 0:   8 + 197 +   5 + 401 + 196 = 807
   Row 1: 740 +   6 +   7 +   3 +  51 = 807
   Row 2:   1 +   2 + 402 +  13 + 389 = 807
   Row 3:   4 + 152 + 279 + 296 +  76 = 807
   Row 4:  54 + 450 + 114 +  94 +  95 = 807

2. COLUMN VECTORS
   Col 0:   8 + 740 +   1 +   4 +  54 = 807
   Col 1: 197 +   6 +   2 + 152 + 450 = 807
   Col 2:   5 +   7 + 402 + 279 + 114 = 807
   Col 3: 401 +   3 +  13 + 296 +  94 = 807
   Col 4: 196 +  51 + 389 +  76 +  95 = 807

3. DIAGONAL VECTORS
   Main Diag (\):   8 +   6 + 402 + 296 +  95 = 807
   Anti Diag (/): 196 +   3 + 402 + 152 +  54 = 807

 STATUS: ALL 12 VECTORS CONFIRMED = 807
